# Export post-fit $z$-expansion parameters to LaTeX

This notebook reloads the selected PROfit chains and generates copy-ready paper text containing the central values and posterior covariance matrices. For fits that also float the interaction-modeling knobs, it appends their pulls as posterior medians with central 68% intervals. Dipole fits are excluded.

Use the controls at the bottom and click **Reload fits and generate LaTeX** after changing a fit input or replacing a ROOT chain.


In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display
import numpy as np

start = Path.cwd().resolve()
NOTEBOOK_DIR = next(
    (path / 'python' / 'scripts' for path in (start, *start.parents)
     if (path / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate python/scripts/postfit_physical_parameters.py')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from postfit_physical_parameters import (
    NUISANCE_BOUNDS, NUISANCE_DISPLAY_NAMES, SPECS, load_fit, nuisance_summary,
)


In [ ]:
SUITES = {
    'NuWro': 'nuwro_fit_results',
    'Open data': 'opendata_fit_results',
    'Asimov': 'asimov_fit_results',
}

# z-expansion fits only. Their ordering is also the output order.
FULL_COVARIANCE_FITS = {
    r'LQCD $k_{\max}=6$': 'lqcd_k6',
    r'LQCD $k_{\max}=7$': 'lqcd_k7',
    r'$\bar{\nu}$–H $k_{\max}=6$': 'minerva_k6',
    r'$\bar{\nu}$–H $k_{\max}=6$ + nuisances': 'minerva_k6_nuisance',
    r'$\bar{\nu}$–H $k_{\max}=6$ uniform': 'minerva_k6_uniform',
    r'$\bar{\nu}$–H $k_{\max}=6$ uniform + nuisances': 'minerva_k6_uniform_nuisance',
    r'$\bar{\nu}$–H $k_{\max}=7$': 'minerva_k7',
    r'$\bar{\nu}$–H $k_{\max}=8$': 'minerva_k8',
    r'$\bar{\nu}$–H + LQCD $k_{\max}=6$': 'minerva_lqcd_k6',
    r'$\bar{\nu}$–H + LQCD $k_{\max}=7$': 'minerva_lqcd_k7',
    r'$\bar{\nu}$–H + LQCD $k_{\max}=6$ + nuisances': 'minerva_lqcd_k6_nuisance',
    r'LQCD $k_{\max}=6$ + nuisances': 'lqcd_k6_nuisance',
}

FIT_LATEX = {
    'lqcd_k6': r'\text{LQCD},\,k_{\max}=6',
    'lqcd_k7': r'\text{LQCD},\,k_{\max}=7',
    'minerva_k6': r'\bar{\nu}\text{--H},\,k_{\max}=6',
    'minerva_k6_nuisance': r'\bar{\nu}\text{--H},\,k_{\max}=6\;\text{with fitted nuisances}',
    'minerva_k6_uniform': r'\bar{\nu}\text{--H},\,k_{\max}=6\;\text{uniform}',
    'minerva_k6_uniform_nuisance': r'\bar{\nu}\text{--H},\,k_{\max}=6\;\text{uniform with fitted nuisances}',
    'minerva_k7': r'\bar{\nu}\text{--H},\,k_{\max}=7',
    'minerva_k8': r'\bar{\nu}\text{--H},\,k_{\max}=8',
    'minerva_lqcd_k6': r'\bar{\nu}\text{--H + LQCD},\,k_{\max}=6',
    'minerva_lqcd_k7': r'\bar{\nu}\text{--H + LQCD},\,k_{\max}=7',
    'minerva_lqcd_k6_nuisance': r'\bar{\nu}\text{--H + LQCD}\;\text{with fitted nuisances}',
    'lqcd_k6_nuisance': r'\text{LQCD},\,k_{\max}=6\;\text{with fitted nuisances}',
}

specs = {spec.key: spec for spec in SPECS}

In [ ]:
def decimal_latex(value, precision):
    value = 0.0 if abs(value) < 0.5 * 10**(-precision) else value
    return f'{value:.{precision}f}'


def signed_latex(value, precision):
    value = 0.0 if abs(value) < 0.5 * 10**(-precision) else value
    prefix = '' if value < 0 else r'\phantom{-}'
    return prefix + f'{value:.{precision}f}'


# The knobs follow the single-parameter convention (posterior median and central 68% interval)
# whatever central value the a coefficients use. The [0, 1] model switches are the fraction of
# the way to the alternative model, so they carry no sigma.
KNOB_LABELS = {display: label for label, display in NUISANCE_DISPLAY_NAMES.items()}


def knob_latex(result, precision):
    summary = nuisance_summary(result)
    if summary is None:
        return None
    entries = []
    for name, row in summary.iterrows():
        label = KNOB_LABELS[name].replace('_', r'\_')
        unit = '' if name in NUISANCE_BOUNDS else r'\,\sigma'
        entries.append(
            rf'$\texttt{{{label}}}={decimal_latex(row.posterior_median, precision)}'
            rf'^{{+{decimal_latex(row.plus_1sigma, precision)}}}'
            rf'_{{-{decimal_latex(row.minus_1sigma, precision)}}}{unit}$'
        )
    return ',\n'.join(entries[:-1]) + ', and\n' + entries[-1]


def central_values(result, convention, independent=True):
    indices = result['joint_indices'] if independent else np.arange(len(result['names']))
    samples = result['joint_samples'] if independent else result['samples']
    if convention == 'Posterior mean':
        return samples.mean(axis=0)
    if convention == 'Posterior median':
        return np.quantile(samples, 0.5, axis=0, method='hazen')
    if convention == 'Profile best fit':
        return result['profile'][indices]
    raise ValueError(f'Unknown central-value convention: {convention}')


def fit_block(
    dataset_label, key, result, convention, precision,
    covariance_precision, full_precision,
):
    names = result['joint_names']
    central = central_values(result, convention)
    covariance = np.atleast_2d(np.cov(result['joint_samples'], rowvar=False))
    full_central = central_values(result, convention, independent=False)
    parameter_list = (
        rf'a_{{1}}, \ldots, a_{{{len(names)}}}' if len(names) > 2
        else ', '.join(rf'a_{{{name[1:]}}}' for name in names)
    )
    central_list = ', '.join(
        decimal_latex(value, precision) for value in central
    )
    full_values = [signed_latex(value, full_precision) for value in full_central]
    full_rows = []
    for start in range(0, len(full_values), 3):
        group = full_values[start:start + 3]
        suffix = ',' if start + 3 < len(full_values) else r'\big).'
        prefix = r'\big(&' if start == 0 else '&'
        full_rows.append(prefix + ','.join(group) + suffix)
    full_central_rows = (r'\\' + '\n               ').join(full_rows)
    covariance_rows = (' ' + r'\\' + '\n').join(
        ' & '.join(signed_latex(value, covariance_precision) for value in row)
        for row in covariance
    )
    fit_name = next(label for label, fit_key in FULL_COVARIANCE_FITS.items()
                    if fit_key == key)
    block = rf'''% {dataset_label}: {fit_name}
we find the fitted parameters to be
\begin{{align}}
    \begin{{pmatrix}}
        {parameter_list}
    \end{{pmatrix}}
    = \begin{{pmatrix}}
        {central_list}
    \end{{pmatrix}},
\end{{align}}
with covariance matrix
\begin{{equation}}
    \begin{{aligned}}
        \begin{{pmatrix}}
        {covariance_rows}
        \end{{pmatrix}},
    \end{{aligned}}
\end{{equation}}
and the full precision coefficients $(a_0,\ldots,a_{len(full_central) - 1})$ are
\begin{{align}}
    \begin{{aligned}}
        {full_central_rows}
    \end{{aligned}}
\end{{align}}'''.strip()
    knobs = knob_latex(result, precision)
    if knobs is not None:
        block += ('\nThe interaction-modeling nuisance parameters are pulled to\n'
                  + knobs + '.')
    return block


def generate_latex(
    dataset_labels, fit_keys, convention, precision, covariance_precision,
    full_precision, burn_in, thin,
):
    blocks = []
    messages = []
    for dataset_label in dataset_labels:
        suite = SUITES[dataset_label]
        blocks.append(f'% ===== {dataset_label} =====')
        for key in fit_keys:
            result = load_fit(
                specs[key], suite, burn_in=burn_in, thin=thin
            )
            if result is None:
                messages.append(
                    f'Skipped {dataset_label} / {key}: no unique ROOT file.'
                )
                continue
            blocks.append(
                fit_block(
                    dataset_label, key, result, convention, precision,
                    covariance_precision, full_precision,
                )
            )
            messages.append(
                f'Loaded {dataset_label} / {key}: '
                f'{len(result["samples"]):,} posterior samples'
            )
    return '\n\n'.join(blocks), messages

## Interactive exporter

In [ ]:
dataset_widget = widgets.SelectMultiple(
    options=list(SUITES), value=tuple(SUITES), description='Datasets',
    rows=len(SUITES), layout=widgets.Layout(width='430px'),
)
fit_widget = widgets.SelectMultiple(
    options=[(label, key) for label, key in FULL_COVARIANCE_FITS.items()],
    value=tuple(FULL_COVARIANCE_FITS.values()), description='Fits',
    rows=len(FULL_COVARIANCE_FITS),
    layout=widgets.Layout(width='430px'),
)
central_widget = widgets.Dropdown(
    options=('Posterior mean', 'Posterior median', 'Profile best fit'),
    value='Posterior mean', description='Central value',
    layout=widgets.Layout(width='430px'),
)
precision_widget = widgets.BoundedIntText(
    value=2, min=0, max=10, description='Fit precision',
    layout=widgets.Layout(width='220px'),
)
covariance_precision_widget = widgets.BoundedIntText(
    value=8, min=1, max=16, description='Cov precision',
    layout=widgets.Layout(width='220px'),
)
full_precision_widget = widgets.BoundedIntText(
    value=8, min=2, max=16, description='Full precision',
    layout=widgets.Layout(width='220px'),
)
burn_in_widget = widgets.BoundedIntText(
    value=0, min=0, max=10_000_000, description='Burn in',
    layout=widgets.Layout(width='220px'),
)
thin_widget = widgets.BoundedIntText(
    value=1, min=1, max=100_000, description='Thin',
    layout=widgets.Layout(width='220px'),
)
generate_button = widgets.Button(
    description='Reload fits and generate LaTeX', icon='refresh',
    button_style='primary', layout=widgets.Layout(width='300px'),
)
status_output = widgets.Output()
latex_output = widgets.Textarea(
    description='LaTeX', layout=widgets.Layout(width='100%', height='600px'),
)

def refresh_latex(_=None):
    with status_output:
        status_output.clear_output(wait=True)
        if not dataset_widget.value or not fit_widget.value:
            print('Select at least one dataset and one fit.')
            return
        try:
            latex, messages = generate_latex(
                dataset_widget.value, fit_widget.value, central_widget.value,
                precision_widget.value, covariance_precision_widget.value,
                full_precision_widget.value,
                burn_in_widget.value, thin_widget.value,
            )
        except Exception as error:
            print(f'{type(error).__name__}: {error}')
            raise
        latex_output.value = latex
        print('\n'.join(messages))
        print(f'\nGenerated {len(latex):,} LaTeX characters. Copy from the box below.')

generate_button.on_click(refresh_latex)
controls = widgets.VBox([
    widgets.HBox([dataset_widget, fit_widget]),
    central_widget,
    widgets.HBox([
        precision_widget, covariance_precision_widget, full_precision_widget,
    ]),
    widgets.HBox([burn_in_widget, thin_widget]),
    generate_button, status_output, latex_output,
])
display(controls)

# Populate all three datasets on the first run.
refresh_latex()